In [1]:
# Optional, run once:
# !pip install torch torchvision wandb

import os
import random
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import wandb

In [2]:
@dataclass
class DataConfig:
    data_dir: str = "./data"
    val_size: int = 5000
    batch_size: int = 1000
    num_workers: int = 0
    pin_memory: bool = True


@dataclass
class ModelConfig:
    num_classes: int = 10
    dropout: float = 0.0


@dataclass
class OptimConfig:
    lr: float = 0.1
    min_lr: float = 0.0
    warmup_epochs: int = 5
    warmup_start_factor: float = 0.01
    momentum: float = 0.9
    weight_decay: float = 3e-4
    nesterov: bool = True


@dataclass
class TrainConfig:
    epochs: int = 100
    amp: bool = True
    grad_clip: float = 0.0
    label_smoothing: float = 0.0
    seed: int = 0
    deterministic: bool = False
    log_every: int = 50


@dataclass
class WandbConfig:
    enabled: bool = True
    project: str = "stable-cifar10"
    run_name: str = "resnet9-baseline-seed0"


@dataclass
class Config:
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    optim: OptimConfig = field(default_factory=OptimConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    wandb: WandbConfig = field(default_factory=WandbConfig)
    out_dir: str = "./runs/resnet9_cifar10_seed0"


cfg = Config()
cfg.train.seed = 0
cfg.wandb.run_name = f"resnet9-baseline-seed{cfg.train.seed}"
cfg.out_dir = f"./runs/resnet9_cifar10_seed{cfg.train.seed}"

cfg

Config(data=DataConfig(data_dir='./data', val_size=5000, batch_size=1000, num_workers=0, pin_memory=True), model=ModelConfig(num_classes=10, dropout=0.0), optim=OptimConfig(lr=0.1, min_lr=0.0, warmup_epochs=5, warmup_start_factor=0.01, momentum=0.9, weight_decay=0.0003, nesterov=True), train=TrainConfig(epochs=100, amp=True, grad_clip=0.0, label_smoothing=0.0, seed=0, deterministic=False, log_every=50), wandb=WandbConfig(enabled=True, project='stable-cifar10', run_name='resnet9-baseline-seed0'), out_dir='./runs/resnet9_cifar10_seed0')

In [3]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(cfg.train.seed)

if cfg.train.deterministic:
    cudnn.benchmark = False
    cudnn.deterministic = True
else:
    cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, pool=False):
        super().__init__()

        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]

        if pool:
            layers.append(nn.MaxPool2d(2))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ResNet9(nn.Module):
    def __init__(self, num_classes=10, dropout=0.0):
        super().__init__()

        self.conv1 = ConvBNReLU(3, 64)
        self.conv2 = ConvBNReLU(64, 128, pool=True)

        self.res1 = nn.Sequential(
            ConvBNReLU(128, 128),
            ConvBNReLU(128, 128),
        )

        self.conv3 = ConvBNReLU(128, 256, pool=True)
        self.conv4 = ConvBNReLU(256, 512, pool=True)

        self.res2 = nn.Sequential(
            ConvBNReLU(512, 512),
            ConvBNReLU(512, 512),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveMaxPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.conv1(x)

        x = self.conv2(x)
        x = x + self.res1(x)

        x = self.conv3(x)
        x = self.conv4(x)
        x = x + self.res2(x)

        x = self.classifier(x)
        return x


model = ResNet9(
    num_classes=cfg.model.num_classes,
    dropout=cfg.model.dropout,
).to(device)

sum(p.numel() for p in model.parameters()) / 1e6

6.57313

In [5]:
def build_dataloaders(cfg: Config):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_full_aug = datasets.CIFAR10(
        root=cfg.data.data_dir,
        train=True,
        download=True,
        transform=train_tf,
    )

    train_full_eval = datasets.CIFAR10(
        root=cfg.data.data_dir,
        train=True,
        download=True,
        transform=eval_tf,
    )

    test_set = datasets.CIFAR10(
        root=cfg.data.data_dir,
        train=False,
        download=True,
        transform=eval_tf,
    )

    n_total = len(train_full_aug)
    n_val = cfg.data.val_size
    n_train = n_total - n_val

    generator = torch.Generator().manual_seed(cfg.train.seed)
    indices = torch.randperm(n_total, generator=generator).tolist()

    train_indices = indices[:n_train]
    val_indices = indices[n_train:]

    train_set = Subset(train_full_aug, train_indices)
    val_set = Subset(train_full_eval, val_indices)

    train_loader = DataLoader(
        train_set,
        batch_size=cfg.data.batch_size,
        shuffle=True,
        num_workers=cfg.data.num_workers,
        pin_memory=cfg.data.pin_memory,
        drop_last=True,
        persistent_workers=cfg.data.num_workers > 0,
    )

    val_loader = DataLoader(
        val_set,
        batch_size=cfg.data.batch_size,
        shuffle=False,
        num_workers=cfg.data.num_workers,
        pin_memory=cfg.data.pin_memory,
        drop_last=False,
        persistent_workers=cfg.data.num_workers > 0,
    )

    test_loader = DataLoader(
        test_set,
        batch_size=cfg.data.batch_size,
        shuffle=False,
        num_workers=cfg.data.num_workers,
        pin_memory=cfg.data.pin_memory,
        drop_last=False,
        persistent_workers=cfg.data.num_workers > 0,
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_dataloaders(cfg)

len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)

c:\Users\luequ\micromamba\envs\torch311\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


(45000, 5000, 10000)

In [6]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.train.label_smoothing)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=cfg.optim.lr,
    momentum=cfg.optim.momentum,
    weight_decay=cfg.optim.weight_decay,
    nesterov=cfg.optim.nesterov,
)

if not 0 <= cfg.optim.warmup_epochs < cfg.train.epochs:
    raise ValueError("warmup_epochs must be in [0, train.epochs)")
if not 0 < cfg.optim.warmup_start_factor <= 1:
    raise ValueError("warmup_start_factor must be in (0, 1]")

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.train.epochs - cfg.optim.warmup_epochs,
    eta_min=cfg.optim.min_lr,
)

if cfg.optim.warmup_epochs > 0:
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=cfg.optim.warmup_start_factor,
        total_iters=cfg.optim.warmup_epochs,
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[cfg.optim.warmup_epochs],
    )
else:
    scheduler = cosine_scheduler

scaler = GradScaler(enabled=cfg.train.amp)

Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)

if cfg.wandb.enabled:
    wandb.init(
        project=cfg.wandb.project,
        name=cfg.wandb.run_name,
        config=asdict(cfg),
    )

C:\tmp\ipykernel_19208\2357173233.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.train.amp)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\luequ\_netrc.
wandb: Currently logged in as: luequinn (luequinn-montana-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
class AverageMeter:
    def __init__(self):
        self.total = 0.0
        self.count = 0

    def update(self, value, n):
        self.total += float(value) * n
        self.count += int(n)

    @property
    def avg(self):
        return self.total / max(1, self.count)


def accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_val_acc):
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "best_val_acc": best_val_acc,
            "cfg": asdict(cfg),
        },
        path,
    )

In [8]:
def train_one_epoch(epoch):
    model.train()

    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    for step, (images, targets) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=cfg.train.amp):
            logits = model(images)
            loss = criterion(logits, targets)

        if cfg.train.amp:
            scaler.scale(loss).backward()

            if cfg.train.grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip)

            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()

            if cfg.train.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip)

            optimizer.step()

        bs = images.size(0)
        acc = accuracy(logits.detach(), targets)

        loss_meter.update(loss.item(), bs)
        acc_meter.update(acc, bs)

        global_step = epoch * len(train_loader) + step

        if cfg.wandb.enabled and step % cfg.train.log_every == 0:
            wandb.log(
                {
                    "train/step_loss": loss.item(),
                    "train/step_acc": acc,
                    "train/lr": optimizer.param_groups[0]["lr"],
                },
                step=global_step,
            )

    return {
        "loss": loss_meter.avg,
        "acc": acc_meter.avg,
    }


@torch.no_grad()
def evaluate(loader):
    model.eval()

    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast(enabled=cfg.train.amp):
            logits = model(images)
            loss = criterion(logits, targets)

        bs = images.size(0)
        acc = accuracy(logits, targets)

        loss_meter.update(loss.item(), bs)
        acc_meter.update(acc, bs)

    return {
        "loss": loss_meter.avg,
        "acc": acc_meter.avg,
    }

In [9]:
best_val_acc = 0.0
best_epoch = -1

for epoch in range(cfg.train.epochs):
    train_metrics = train_one_epoch(epoch)
    val_metrics = evaluate(val_loader)

    scheduler.step()

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch

    log_dict = {
        "epoch": epoch,
        "train/loss": train_metrics["loss"],
        "train/acc": train_metrics["acc"],
        "val/loss": val_metrics["loss"],
        "val/acc": val_metrics["acc"],
        "lr": optimizer.param_groups[0]["lr"],
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
    }

    if cfg.wandb.enabled:
        global_step = (epoch + 1) * len(train_loader) - 1
        wandb.log(log_dict, step=global_step)

    print(
        f"epoch={epoch:03d} "
        f"train_loss={train_metrics['loss']:.4f} "
        f"train_acc={train_metrics['acc']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} "
        f"val_acc={val_metrics['acc']:.4f} "
        f"best_val_acc={best_val_acc:.4f}"
    )

C:\tmp\ipykernel_19208\597135703.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.amp):
C:\tmp\ipykernel_19208\597135703.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.amp):


epoch=000 train_loss=2.2128 train_acc=0.3133 val_loss=1.6905 val_acc=0.3928 best_val_acc=0.3928
epoch=001 train_loss=7.5765 train_acc=0.1307 val_loss=2.7941 val_acc=0.1148 best_val_acc=0.3928
epoch=002 train_loss=3.0812 train_acc=0.1286 val_loss=2.8410 val_acc=0.1238 best_val_acc=0.3928
epoch=003 train_loss=3.1231 train_acc=0.1923 val_loss=2.6317 val_acc=0.1770 best_val_acc=0.3928


c:\Users\luequ\micromamba\envs\torch311\Lib\site-packages\torch\optim\lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


epoch=004 train_loss=3.4591 train_acc=0.2061 val_loss=2.2134 val_acc=0.1882 best_val_acc=0.3928
epoch=005 train_loss=2.0057 train_acc=0.2859 val_loss=1.8770 val_acc=0.2674 best_val_acc=0.3928
epoch=006 train_loss=1.7123 train_acc=0.3762 val_loss=1.6002 val_acc=0.4042 best_val_acc=0.4042
epoch=007 train_loss=1.5807 train_acc=0.4307 val_loss=1.5729 val_acc=0.4484 best_val_acc=0.4484
epoch=008 train_loss=1.4235 train_acc=0.4906 val_loss=1.4836 val_acc=0.4964 best_val_acc=0.4964
epoch=009 train_loss=1.2652 train_acc=0.5466 val_loss=1.2554 val_acc=0.5492 best_val_acc=0.5492
epoch=010 train_loss=1.1319 train_acc=0.5971 val_loss=1.1768 val_acc=0.5914 best_val_acc=0.5914
epoch=011 train_loss=0.9704 train_acc=0.6559 val_loss=0.8712 val_acc=0.7000 best_val_acc=0.7000
epoch=012 train_loss=0.8586 train_acc=0.6981 val_loss=0.9404 val_acc=0.6860 best_val_acc=0.7000
epoch=013 train_loss=0.7331 train_acc=0.7420 val_loss=0.8927 val_acc=0.6976 best_val_acc=0.7000
epoch=014 train_loss=0.6520 train_acc=0.

KeyboardInterrupt: 

In [ ]:
best_path = os.path.join(cfg.out_dir, "best.pt")

ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model"])

test_metrics = evaluate(test_loader)

print(
    f"test_loss={test_metrics['loss']:.4f} "
    f"test_acc={test_metrics['acc']:.4f} "
    f"best_epoch={ckpt['epoch']}"
)

if cfg.wandb.enabled:
    wandb.log(
        {
            "test/loss": test_metrics["loss"],
            "test/acc": test_metrics["acc"],
            "test/best_epoch": ckpt["epoch"],
        },
        step=cfg.train.epochs * len(train_loader),
    )
    wandb.finish()

---
# Metasmoothness: is this training routine amenable to metagradients?

Everything above trains the model **once**. This section measures, and then
maximizes, the **metasmoothness** of the training routine, following Engstrom et
al. (2025), *Optimizing ML Training with Metagradient Descent*
([arXiv:2503.13751](https://arxiv.org/abs/2503.13751)).

**Why it matters.** If we treat data curation / hyperparameters as a continuous
optimization problem over a *metaparameter* $z$, the **metagradient**
$\nabla_z\,\phi(\mathcal{A}(z))$ points toward better training setups. But
metagradients are only useful if the training function $f=\phi\circ\mathcal{A}$
is *smooth* in $z$. The paper makes this precise with two cheap, **metagradient-free**
diagnostics — each needs only **three deterministic training runs** at $z$,
$z+hv$, $z+2hv$:

- **Definition 1 — curvature**
  $\;S_{h,v}(f;z)=\dfrac{\lvert f(z+2hv)-2f(z+hv)+f(z)\rvert}{h^2}$, a second-order
  finite difference along $v$.  **Lower = smoother** (a $\beta$-smooth $f$ has $S\le\beta$).
- **Definition 2 — empirical metasmoothness**
  $\;\hat S_{h,v}(\mathcal{A};z)\in[-1,1]$: the range-weighted **sign agreement**
  between consecutive finite-difference metagradients in *parameter* space.
  **Higher ($\to 1$) = smoother.**

**Setup.** The output function $\phi$ is the **held-out classification loss**
(cross-entropy on a disjoint CIFAR-10 split). The metaparameter $z$ is a vector
of **continuous data weights at $z=0$** (the paper's count relaxation, Sec. 4.1.2):
example $i$ is weighted by $\exp(z_i)$, so $z=0$ is ordinary training. We analyse
**both** groupings this project cares about: **per-cluster** (one weight per CIFAR
class) and **per-example** (one weight per image). All metric machinery lives in
[`metasmooth.py`](metasmooth.py); see [`tests/test_metasmooth.py`](tests/test_metasmooth.py).

In [ ]:
import importlib

import numpy as np
import matplotlib.pyplot as plt
import torch

import metasmooth as ms
importlib.reload(ms)

bench_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# phi = held-out cross-entropy; z = continuous per-(cluster|example) weights at 0.
subset = ms.load_cifar_subset(cfg.data.data_dir, n_train=3000, n_val=1000,
                              seed=0, device=bench_device)

# Each probe is 3 deterministic fp32 trainings (amp="off" keeps the tiny
# Definition-1 second difference resolvable; fp16/bf16 are NOT faithful here).
# Scale up n_train / epochs / num_directions on a faster machine -- the metric
# and the conclusions are unchanged.
bench_cfg = ms.TrainConfig(epochs=16, batch_size=500, lr=0.08, amp="off")
H, NUM_DIRECTIONS = 0.05, 3

base_params = sum(p.numel() for p in ms.ResNet9(ms.BASELINE_ROUTINE).parameters())
print(f"device={bench_device}  n_train={subset.n_train}  n_val={subset.n_val}  "
      f"epochs={bench_cfg.epochs}  lr={bench_cfg.lr}  "
      f"baseline_params={base_params/1e6:.2f}M")


def run_benchmark(routine, kind, ndirs=NUM_DIRECTIONS):
    mp = ms.make_metaparam(kind, subset)
    return ms.benchmark_metasmoothness(
        subset, mp, routine, bench_cfg, h=H, num_directions=ndirs,
        direction_seed=1000, device=bench_device,
    )

## 1. Benchmark the current (baseline) routine

The notebook's ResNet-9 uses **max pooling**, BatchNorm-before-ReLU,
full-magnitude logits, and base width. We measure its metasmoothness at $z=0$,
averaged over several random probe directions $v\sim\mathcal{N}(0,I)$ (the same
directions are reused across routines so comparisons are apples-to-apples).

In [ ]:
baseline = {k: run_benchmark(ms.BASELINE_ROUTINE, k) for k in ("per_cluster", "per_example")}
for r in baseline.values():
    print(r.summary())

## 2. Maximize metasmoothness — the paper's design menu

The paper makes a learning algorithm metasmooth by exploring a fixed menu of
modifications and keeping those that raise empirical metasmoothness (Sec. 3.2,
Remark 4). Each maps to a knob on `ms.Routine`:

| paper modification | knob | smooth choice |
|---|---|---|
| average instead of **max pooling** | `pool` | `"avg"` |
| **scale down the last layer's output** (~10x) | `final_scale` | `10.0` |
| BatchNorm **before** the activation | `bn_before_act` | `True` *(baseline already does this)* |
| **wider** network | `width` | `> 1` |
| *(extra)* smooth activation (GELU vs ReLU) | `smooth_act` | `True` |

The smooth optimizer recurrence (eps inside the sqrt) is already used by the
project's functional AdamW. `SMOOTH_ROUTINE` applies the **capacity-matched**
architectural levers (avg pool + BN-before-act + final/10, width unchanged);
`SMOOTH_WIDE_ROUTINE` adds the 2x width lever.

In [ ]:
print("baseline   :", ms.BASELINE_ROUTINE)
print("smooth     :", ms.SMOOTH_ROUTINE)
print("smooth_wide:", ms.SMOOTH_WIDE_ROUTINE)

In [ ]:
smooth = {k: run_benchmark(ms.SMOOTH_ROUTINE, k) for k in ("per_cluster", "per_example")}
for r in smooth.values():
    print(r.summary())

print("\nbaseline -> smooth (same capacity, same data, same probe directions):")
for k in ("per_cluster", "per_example"):
    b, s = baseline[k], smooth[k]
    print(f"  {k:12s}  S(Def1) {b.s_curvature_mean:7.2f} -> {s.s_curvature_mean:7.2f}"
          f"   S_hat(Def2) {b.s_hat_mean:+.3f} -> {s.s_hat_mean:+.3f}"
          f"   val_loss {b.f0:.3f} -> {s.f0:.3f}")

## 3. Which modification matters most? (per-cluster ablation)

Turn each knob on **individually** (vs the baseline), including the deliberately
*non-smooth* `bn_after_act` direction, then the full smooth routines.

In [ ]:
menu = [
    ms.BASELINE_ROUTINE,
    ms.Routine(pool="avg", name="avg_pool"),
    ms.Routine(final_scale=10.0, name="final/10"),
    ms.Routine(bn_before_act=False, name="bn_after_act"),  # non-smooth direction
    ms.Routine(width=2.0, name="width_x2"),
    ms.SMOOTH_ROUTINE,
    ms.SMOOTH_WIDE_ROUTINE,
]
ablation = [run_benchmark(r, "per_cluster", ndirs=2) for r in menu]
for r in ablation:
    print(r.summary())

In [ ]:
names = [r.routine for r in ablation]
shat = [r.s_hat_mean for r in ablation]
scur = [r.s_curvature_mean for r in ablation]
colors = ["#2a9d8f" if n in ("smooth", "smooth_wide") else
          ("#e76f51" if n == "bn_after_act" else "#5a6472") for n in names]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(names, shat, color=colors); axes[0].axhline(0, color="k", lw=.6)
axes[0].set_title(r"empirical metasmoothness $\hat S$ (Def 2) — higher = smoother")
axes[1].bar(names, scur, color=colors)
axes[1].set_title(r"curvature $S$ (Def 1) — lower = smoother")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
plt.tight_layout(); plt.show()

## Results & conclusions

*Laptop-scale run (`artifacts/metasmooth_results.json`, produced by
`python -m experiments.cifar10.metasmooth_benchmark`): `n_train=3000`,
`n_val=1000`, 16 epochs, **fp32**, `h=0.05`, 3 probe directions (2 for the
ablation). The metric is **relative** — scale up `n_train` / `epochs` /
`num_directions` on the VM; the ordering is what matters.*

**Baseline vs the paper's menu** (held-out cross-entropy as $\phi$; smooth routine is **capacity-matched**):

| metaparameter $z$ | routine | val loss $f_0$ | $S$ (Def 1) ↓ | $\hat S$ (Def 2) ↑ |
|---|---|---|---|---|
| per-cluster (class) | baseline | 2.078 | 53.1 | +0.092 |
| per-cluster (class) | **smooth** | **1.425** | **4.8** | **+0.511** |
| per-example | baseline | 2.078 | 43.4 | +0.135 |
| per-example | **smooth** | **1.425** | **7.7** | +0.148 |

**Per-cluster ablation** (one knob at a time):

| routine | $S$ (Def 1) ↓ | $\hat S$ (Def 2) ↑ |
|---|---|---|
| baseline (max pool, full-magnitude logits) | 53.1 | +0.092 |
| average pooling only | 30.2 | +0.073 |
| **final-layer output ÷10 only** | **2.3** | +0.123 |
| **smooth (avg pool + ÷10 + BN-before-act)** | 4.8 | **+0.511** |

![metasmoothness results](artifacts/metasmooth_results.png)

**Takeaways**

1. **The baseline routine is *not* metasmooth.** Curvature $S\approx 43\text{–}53$
   and empirical metasmoothness $\hat S\approx +0.09\text{–}0.14$ (≈0 means
   consecutive finite-difference metagradients barely agree). Metagradients on
   this routine would be unreliable — exactly the failure mode the paper warns
   about and motivates "smooth model training" to fix.
2. **The paper's menu fixes it.** The capacity-matched smooth routine cuts
   curvature **~6–11×** (per-cluster $53\to4.8$, per-example $43\to7.7$) and, for
   per-cluster weights, raises $\hat S$ from **+0.09 to +0.51** on the paper's
   preferred parameter-space metric.
3. **The dominant lever is scaling the final-layer output down (÷10):** alone it
   collapses $S$ from 53 to 2.3. Average pooling helps the curvature modestly
   (and noisily); the two are **complementary for $\hat S$** — only the
   *combination* pushes $\hat S$ to +0.51. The notebook's ResNet-9 **already uses
   BatchNorm-before-activation** (the paper's smooth placement), so that lever was
   on from the start. (Width is an extra paper lever — run the ablation with
   `ms.SMOOTH_WIDE_ROUTINE` / `Routine(width=2.0)` to measure it here.)
4. **Per-example $\hat S$ barely moves even though $S$ drops sharply.** With one
   weight per image (3000-dim $z$) each coordinate perturbs a single example by
   ~1%, so per-parameter changes are tiny and their signs are noisy — the
   sign-agreement average $\hat S$ sits near its noise floor regardless of
   routine. In the per-example regime the scalar curvature $S$ is the more
   informative metric, and it improves clearly.
5. **Smoothness tracks optimizability.** The smooth routine also reaches a lower
   held-out loss ($2.08\to1.42$; final/10 alone $\to0.99$), matching the paper's
   observation that metasmooth routines are easier to optimize.

**Recommendation.** To make this routine amenable to metagradient data curation,
modify the ResNet-9 classifier to **scale its logits down ~10×** and **use
average (not max) pooling**, keeping BatchNorm-before-activation (optionally
widen): `ms.SMOOTH_ROUTINE` / `ms.SMOOTH_WIDE_ROUTINE` in
[`metasmooth.py`](metasmooth.py).

**Reproduce / scale up.** Re-run the cells above, or run
`python -m experiments.cifar10.metasmooth_benchmark` (writes
`artifacts/metasmooth_results.json`; render with `python -m experiments.cifar10.render_metasmooth`).
Keep `amp="off"`: fp16/bf16 flip the sign of the tiny Definition-1 second
difference. Increase `n_train`, `epochs`, and `num_directions` for tighter
estimates.